# Data Preparation
**ResolveX — M2: Data Preparation & EDA**

This notebook builds the three final datasets required for downstream ML and retrieval development.

Preprocessing logic is imported from `src/data/preprocessing.py`.

**Outputs written to `data/processed/`:**
- `ml_dataset.csv` — single-category questions for ML classification.
- `knowledge_base.jsonl` — questions and answers with preserved technical content for RAG.
- `retrieval_eval.csv` — duplicate-question pairs from PostLinks.xml for retrieval evaluation.

## 1. Setup

Import dependencies and configure paths. Preprocessing logic is imported from `src/data/preprocessing.py`.

In [1]:
from pathlib import Path
import sys
import csv
import json

cwd = Path.cwd().resolve()
ROOT = next((folder for folder in (cwd, *cwd.parents)
             if (folder / "src" / "data" / "preprocessing.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook within the ResolveX repository")
sys.path.insert(0, str(ROOT))

QUESTIONS   = ROOT / "data" / "interim"    / "questions.csv"
ANSWERS     = ROOT / "data" / "interim"    / "answers.csv"
POSTLINKS   = ROOT / "data" / "raw"        / "PostLinks.xml"
ML_OUT      = ROOT / "data" / "processed"  / "ml_dataset.csv"
KB_OUT      = ROOT / "data" / "processed"  / "knowledge_base.jsonl"
EVAL_OUT    = ROOT / "data" / "processed"  / "retrieval_eval.csv"

from src.data.preprocessing import prepare_datasets

## 2. Build the ML Dataset

Extract single-category questions from the interim CSV.

Each row contains: `Id`, `Title`, `BodyText` (HTML cleaned, code blocks preserved), `Tags` (metadata only, not an input feature), and `Label`.

Only questions matched to exactly one ResolveX category are included. Multi-category and unmatched questions are excluded from this dataset.

In [2]:
SUMMARY = ML_OUT.parent / "preparation_summary.json"
if not SUMMARY.exists():
    summary = prepare_datasets(ROOT)
else:
    for path in (ML_OUT, KB_OUT, EVAL_OUT, ML_OUT.parent / "duplicate_relationships.csv"):
        if not path.is_file():
            raise FileNotFoundError(f"Missing prepared file: {path}. Rebuild with prepare_datasets(ROOT, overwrite=True).")
    summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
print("Prepared ML rows:", summary["ml_rows"])
# To deliberately regenerate all outputs after preprocessing changes:
# summary = prepare_datasets(ROOT, overwrite=True)


Prepared ML rows: 177194


### 2.1 Verify the ML Dataset

Confirm row count, label distribution, and that no Tags column is missing.

In [3]:
from collections import Counter

label_counts = Counter()
missing_title = 0
missing_body  = 0
total         = 0

with ML_OUT.open(encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        total += 1
        label_counts[row["Label"]] += 1
        missing_title += not bool(row.get("Title", "").strip())
        missing_body  += not bool(row.get("BodyText", "").strip())

print(f"Total rows : {total:,}")
print(f"Missing titles : {missing_title}")
print(f"Missing body text : {missing_body}")
print("\nLabel distribution:")
for label, n in sorted(label_counts.items()):
    print(f"  {label:<30} {n:>7,}")

Total rows : 177,194
Missing titles : 0
Missing body text : 0

Label distribution:
  Access & Permissions            14,836
  Boot & Startup                  27,048
  Hardware & Drivers              43,777
  Network & Connectivity          30,107
  Software & Packages             43,585
  Storage & Filesystems           17,841


## 3. Build the Knowledge Base

Build a JSON Lines file where each record contains a question, its linked answers, cleaned body text, and attribution metadata.

Invalid accepted-answer references (9 missing + 2 mismatched ParentId, documented in M1) are excluded and counted. Original records in the interim CSVs are not modified.

In [4]:
print("Knowledge-base preparation summary:")
print(summary["knowledge_base"])


Knowledge-base preparation summary:
{'questions': 318537, 'answers': 523728, 'invalid_accepted': 11, 'missing_accepted': 9, 'mismatched_accepted': 2, 'answer_count_mismatches': 87, 'questions_without_answers': 95914}


### 3.1 Verify the Knowledge Base

Sample the first record to confirm structure, attribution fields, and code block preservation.

In [5]:
with KB_OUT.open(encoding="utf-8") as f:
    sample = json.loads(f.readline())

print("Fields:", list(sample.keys()))
print("Answer fields:", list(sample["answers"][0].keys()) if sample["answers"] else "no answers")
print("\nSample question ID:", sample["question_id"])
print("Title:", sample["title"])
print("Accepted answer ID:", sample["accepted_answer_id"])
print("Content license:", sample["content_license"])
print("Number of answers:", len(sample["answers"]))
print("\nBody preview (first 300 chars):")
print(sample["body_text"][:300])

Fields: ['question_id', 'source_url', 'body_html', 'recorded_answer_count', 'actual_answer_count', 'recorded_accepted_answer_id', 'title', 'body_text', 'tags', 'creation_date', 'score', 'accepted_answer_id', 'content_license', 'owner_user_id', 'answers']
Answer fields: ['answer_id', 'source_url', 'body_html', 'body_text', 'score', 'creation_date', 'is_accepted', 'content_license', 'owner_user_id']

Sample question ID: 1
Title: How to get the "Your battery is broken" message to go away?
Accepted answer ID: 2
Content license: CC BY-SA 3.0
Number of answers: 4

Body preview (first 300 chars):
Every time I turn on my computer, I see a message saying something like:



```
Your battery may be old or broken.

```



I am already aware that my battery is bad. How do I suppress this message?


## 4. Build Retrieval Evaluation Data

Extract duplicate-question pairs from `PostLinks.xml` (LinkTypeId = 3).

Queries must exist in the interim questions CSV; targets must exist in the knowledge base. Self matches must be excluded during ranking. Full duplicate relationships are retained separately for classifier split grouping. These pairs are used to evaluate retrieval quality: given a question, the duplicate should appear in the top-K retrieved results.

In [6]:
print("Full duplicate relationships:", summary["duplicate_relationships"])
print("Retrievable-target evaluation pairs:", summary["retrieval_pairs"])
print("Targets absent from KB:", summary["duplicate_relationships"] - summary["retrieval_pairs"])


Full duplicate relationships: 41931
Retrievable-target evaluation pairs: 41411
Targets absent from KB: 520


### 4.1 Verify the Retrieval Evaluation Data

Confirm pair count and check for duplicate or self-referential entries.

In [7]:
pairs      = []
self_links = 0

with EVAL_OUT.open(encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        pairs.append((row["QuestionId"], row["RelatedQuestionId"]))
        if row["QuestionId"] == row["RelatedQuestionId"]:
            self_links += 1

unique_pairs = len(set(pairs))
print(f"Total pairs     : {len(pairs):,}")
print(f"Unique pairs    : {unique_pairs:,}")
print(f"Self-links      : {self_links}")
print(f"Duplicate rows  : {len(pairs) - unique_pairs}")

Total pairs     : 41,411
Unique pairs    : 41,411
Self-links      : 0
Duplicate rows  : 0


## Findings

### ML Dataset

- Single-category questions extracted from 177,194 candidates.
- Each row contains cleaned body text with code, URLs and image descriptions preserved. CreationDate is retained for temporal analysis. Labels remain weak supervision.
- Tags are retained as metadata; they are not input features.

### Knowledge Base

- Questions with at least one linked answer are included.
- All invalid accepted-answer references are counted before excluding unanswered questions. Recorded values are retained alongside valid links and actual answer counts.
- Attribution fields (ContentLicense, OwnerUserId, CreationDate, post IDs) are preserved on every record.

### Retrieval Evaluation Data

- Duplicate-question pairs extracted from PostLinks.xml (LinkTypeId = 3).
- Queries exist in the question dataset and targets exist in the knowledge base. Exclude each query ID from retrieval results; group relevant targets per query for Recall@K and MRR.
- These pairs will be used to measure Recall@K and MRR in M4.

## Preparation decisions and limits

- Parse HTML while preserving code indentation, commands, inline code, link destinations and image descriptions. Keep original HTML in the knowledge base; images are not processed with OCR.
- Preserve text case and punctuation; do not truncate or balance the dataset during preparation.
- Use actual linked answers for knowledge-base inclusion. Retain recorded counts and accepted references separately from validated relationships. The run found 87 count discrepancies and 11 invalid accepted references (9 missing answers, 2 wrong parents).
- Keep all 41,931 duplicate relationships for later split grouping. The retrieval benchmark retains 41,411 pairs with targets in the KB, excluding 520 unavailable targets.
- Labels are tag-derived candidates, not verified ground truth. Initial inspection found mislabeled and overlapping cases. Community answers may be outdated and are not guaranteed solutions.
- In M3, review label quality and keep duplicate-connected questions in the same split. In M4, resolve author attribution for display and exclude each query itself from retrieval results.

### Validation

Both notebooks completed successfully. Full-output checks found no empty derived bodies or duplicate KB IDs and verified accepted relationships and retrieval-target membership. Decoded content was checked in 443,729 answer code blocks without nested markup; 725 blocks with nested markup were excluded from that exact-content check and retain their original HTML. These checks establish data consistency, not answer correctness.

To regenerate outputs deliberately after preprocessing changes, use `prepare_datasets(ROOT, overwrite=True)`, then rerun notebooks 03 and 04.
